# 🔬 Arm 1 (Priority 1 — Primary): Invariance-Regularized Policy Optimization (Inv-GRPO)
**Project:** Diagnosing and Resolving Code Small Language Model Mimicry via a Reduction Ladder  
**Organization:** Orange Innovation Labs — AI Research & Development Division  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada Soliman  

---

## 🎯 Theoretical Foundation
When Small Language Models (~1.5B parameters) are trained with standard binary RLVR (Reinforcement Learning with Verifiable Rewards), they succumb to **shortcut mimicry**: memorizing canonical $L_0$ templates and reproducing them blindly on surface-perturbed tasks $L_1–L_5$.

**Inv-GRPO** is our primary innovation at Orange Innovation Labs.  
Instead of evaluating prompts in isolation, Inv-GRPO feeds **paired semantically invariant prompts** $(x, x')$ during each rollout group:
- $x$: Canonical $L_0$ task (HumanEval)
- $x'$: Perturbed counterpart (e.g., $L_2$ EvoEval ToolUse)

### Mathematical Formulation (from Research Proposal §8.1):
$$\mathcal{R}_{\text{total}}(y_i, y'_i) = \mathcal{R}_{\text{exec}}(y_i) + \mathcal{R}_{\text{exec}}(y'_i) + \lambda \cdot \mathcal{R}_{\text{consistency}}(y_i, y'_i) - \gamma \cdot \mathcal{P}_{\text{template}}(y'_i)$$

| Symbol | Meaning | Role in Optimization |
|---|---|---|
| $\mathcal{R}_{\text{exec}}(y) \in \{0,1\}$ | Sandbox unit-test pass/fail | Solves the specific task representation |
| $\mathcal{R}_{\text{consistency}}(y, y')$ | Bonus when **both** $(x, x')$ pass | Enforces semantic invariance across representations |
| $\mathcal{P}_{\text{template}}(y')$ | Penalty for verbatim shortcut on perturbed task | Actively unlearns memorized textbook shortcuts |
| $\hat{A}_i$ | Normalized group advantage | Guides policy gradient updates without a Critic model |

> 📌 **Key Architectural Advantage:** **Zero Inference Latency Overhead!**  
> Regularization occurs strictly at train-time. At inference, Model M6 takes standard individual prompts at full speed.

---
## 1. Environment & Hardware Initialization

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

from src.core.config import get_settings
from src.arms.arm1_inv_grpo import (
    PairedTask,
    InvGRPODatasetLoader,
    InvGRPORewardEngine,
    InvGRPOTrainer,
    InvGRPOEvaluator,
)

settings = get_settings()

print("✅ Inv-GRPO Environment Initialized.")
print(f"   CUDA Available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device            : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   Total VRAM            : {vram:.2f} GB")
print(f"   Base Student Model    : {settings.models.student_model}")
print(f"   Group Size (G)        : {settings.inv_grpo.group_size}")
print(f"   Lambda Consistency    : {settings.inv_grpo.lambda_consistency}")
print(f"   Gamma Template Penalty: {settings.inv_grpo.gamma_template_penalty}")

---
## 2. Paired Invariance Dataset Loader ($L_0$ vs $L_2$)

We pair canonical $L_0$ tasks (HumanEval) with their $L_2$ counterparts (EvoEval ToolUse).  
Each paired task includes:
1. `prompt_orig` & `test_orig`: Canonical problem and assertions.
2. `prompt_pert` & `test_pert`: Perturbed problem with API helper function and assertions.
3. `decoy_code`: Canonical shortcut template to detect and penalize if blindly pasted.

In [ ]:
loader = InvGRPODatasetLoader(perturbed_level="L2")
train_pairs, test_pairs = loader.load_pairs(max_pairs=30, train_ratio=0.8, seed=settings.project.seed)

print(f"✅ Paired Tasks Loaded:")
print(f"   Training Pairs   : {len(train_pairs)}")
print(f"   Evaluation Pairs : {len(test_pairs)}")

sample = train_pairs[0]
print("\n--- SAMPLE PAIRED TASK ---")
print(f"Pair ID        : {sample.pair_id}")
print(f"L0 Canonical ID: {sample.l0_task_id} (Entry: {sample.entry_orig})")
print(f"L2 Perturbed ID: {sample.pert_task_id} (Entry: {sample.entry_pert})")
print(f"\n[Canonical Prompt L0]:\n{sample.prompt_orig[:250]}...")
print(f"\n[Perturbed Prompt L2]:\n{sample.prompt_pert[:250]}...")
print(f"\n[Decoy Shortcut to Penalize]:\n{sample.decoy_code[:200]}...")

---
## 3. Multi-Objective Invariance Reward Engine Verification

Before full training, we verify the reward mechanics using three characteristic candidate behaviors:
1. **Candidate 1 (Shortcut Mimic):** Verbatim copies the $L_0$ shortcut onto $L_2$ → **Fails & Penalized (-0.5)**.
2. **Candidate 2 (Partial Hit):** Solves the perturbed task but fails the canonical task → **No Consistency Bonus**.
3. **Candidate 3 (Invariant Reasoner):** Solves both representations correctly → **Earns Consistency Bonus (+0.5)**.

In [ ]:
reward_engine = InvGRPORewardEngine(
    lambda_consistency=settings.inv_grpo.lambda_consistency,
    gamma_template_penalty=settings.inv_grpo.gamma_template_penalty,
)

# Simulation candidates on a paired problem
sim_candidates = [
    {
        "label": "Cand 1: Shortcut Mimic (Verbatim Copy)",
        "solution_orig": "def add(a, b): return a + b",
        "solution_pert": "def add(a, b): return a + b",  # Wrong entry point → fails & mimicked decoy!
    },
    {
        "label": "Cand 2: Partial Hit (Only perturbed)",
        "solution_orig": "def add(a, b): return a * b",   # Bug in canonical
        "solution_pert": "def add_str(s): return sum(map(int, s.split(',')))",
    },
    {
        "label": "Cand 3: Invariant Reasoner (Both correct)",
        "solution_orig": "def add(a, b): return a + b",
        "solution_pert": "def add_str(s): return sum(map(int, s.split(',')))",
    },
]

dummy_task = PairedTask(
    pair_id="sim_test",
    l0_task_id="Sim/Add",
    pert_task_id="Sim/AddStr",
    ladder_level="L2",
    prompt_orig="def add(a, b):\n    \"\"\"Return a + b\"\"\"\n",
    test_orig="assert add(2, 3) == 5\nassert add(-1, 1) == 0",
    entry_orig="add",
    canonical_orig="def add(a, b): return a + b",
    prompt_pert="def add_str(s):\n    \"\"\"Return sum of comma-separated ints\"\"\"\n",
    test_pert="assert add_str('2,3') == 5\nassert add_str('-1,1') == 0",
    entry_pert="add_str",
    canonical_pert="def add_str(s): return sum(map(int, s.split(',')))",
    decoy_code="def add(a, b): return a + b",
)

advs, sim_evals = reward_engine.compute_group_advantages(sim_candidates, dummy_task)

sim_df = pd.DataFrame([
    {"Candidate": c["label"], "Advantage": round(float(adv), 2), **ev}
    for c, adv, ev in zip(sim_candidates, advs, sim_evals)
])

print("📊 Inv-GRPO Advantage Separation on Simulated Candidates:")
print(sim_df[["Candidate", "r_total", "r_consistency", "p_template", "Advantage"]].to_string(index=False))

# Plot Advantage Bar Chart
plt.figure(figsize=(7, 3.5), dpi=140)
bars = plt.bar(
    ["Shortcut Mimic", "Partial Hit", "Invariant Reasoner"],
    advs,
    color=["#DC3545", "#FFC107", "#28A745"],
    edgecolor="black",
    linewidth=0.5,
)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
for bar, val in zip(bars, advs):
    offset = 0.05 if val >= 0 else -0.15
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
             f"{val:+.2f}", ha="center", va="bottom", fontweight="bold", fontsize=9)
plt.title(r"Inv-GRPO Normalized Advantage Separation ($\hat{A}_i$)", fontsize=11, fontweight="bold")
plt.ylabel("Advantage Value", fontsize=10)
plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/inv_grpo_advantage.png", dpi=140)
plt.show()
print("Saved → results/inv_grpo_advantage.png")

---
## 4. Model Initialization (Model M6 Initializer with 4-bit LoRA)

We load `Qwen/Qwen2.5-Coder-1.5B-Instruct` in 4-bit NF4 precision with PEFT/LoRA.
This uses ~1.5 GB of VRAM, fitting easily within the RTX 3070 Ti (8 GB) envelope.

In [ ]:
trainer = InvGRPOTrainer(
    output_dir="checkpoints/inv_grpo_final",
    group_size=settings.inv_grpo.group_size,
    learning_rate=settings.inv_grpo.learning_rate,
    max_new_tokens=96,
    temperature=0.8,
    lambda_consistency=settings.inv_grpo.lambda_consistency,
    gamma_template_penalty=settings.inv_grpo.gamma_template_penalty,
)

print(f"\n✅ Model M6 ready on {trainer.device}.")
if torch.cuda.is_available():
    alloc_vram = torch.cuda.memory_allocated() / (1024**3)
    print(f"   Current Allocated VRAM: {alloc_vram:.2f} GB")

---
## 5. Inv-GRPO Online Policy Optimization Loop

For each training iteration:
1. **Rollout:** Sample $G=4$ completions for canonical task $x$ and $G=4$ completions for perturbed task $x'$.
2. **Sandbox Scoring:** Evaluate all pairs in isolated sub-processes.
3. **Invariance Advantage:** Compute $\hat{A}_i = (\mathcal{R}_i - \bar{\mathcal{R}}) / (\sigma + \epsilon)$.
4. **Policy Gradient Step:** Backpropagate surrogate loss $-\frac{1}{G} \sum \hat{A}_i [\log \pi(y \mid x) + \log \pi(y' \mid x')]$ into LoRA weights.

In [ ]:
# Execute Inv-GRPO training loop on the paired training benchmark set
NUM_TRAIN_STEPS = 12  # 12 steps: optimal convergence & speed on RTX 3070 Ti

history = trainer.train(
    train_pairs=train_pairs,
    num_steps=NUM_TRAIN_STEPS,
    grad_accum_steps=2,
)

print("\n🎉 Inv-GRPO Training Completed Successfully!")

---
## 6. Training Dynamics & Invariance Convergence Visualization

We plot three core metrics across training steps:
1. **Mean Total Invariance Reward:** Tracking overall solution quality.
2. **Pairwise Consistency Rate (%):** Tracking genuine cross-representation transfer.
3. **Template Penalty Rate (%):** Demonstrating the active unlearning of shortcut mimicry.

In [ ]:
steps = history["step"]
rewards = history["mean_reward"]
cons_rates = history["consistency_rate"]
pen_rates = history["template_penalty_rate"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), dpi=140)

# 1. Mean Reward
axes[0].plot(steps, rewards, marker="o", color="#FF6400", linewidth=1.8, label="Total Reward")
axes[0].set_title("Mean Invariance Reward (R_total)", fontweight="bold", fontsize=10)
axes[0].set_xlabel("Training Step", fontsize=9)
axes[0].set_ylabel("Reward Value", fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Consistency Bonus Rate
axes[1].plot(steps, cons_rates, marker="s", color="#28A745", linewidth=1.8, label="Consistency %")
axes[1].set_title("Pairwise Consistency Rate (%)", fontweight="bold", fontsize=10)
axes[1].set_xlabel("Training Step", fontsize=9)
axes[1].set_ylabel("Both Solved (%)", fontsize=9)
axes[1].set_ylim(-5, 105)
axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. Shortcut Penalty Rate (Unlearning Mimicry)
axes[2].plot(steps, pen_rates, marker="^", color="#DC3545", linewidth=1.8, label="Shortcut Penalty %")
axes[2].set_title("Shortcut Mimicry Penalty Rate (%)", fontweight="bold", fontsize=10)
axes[2].set_xlabel("Training Step", fontsize=9)
axes[2].set_ylabel("Mimicked Decoy (%)", fontsize=9)
axes[2].set_ylim(-5, 105)
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/inv_grpo_training_dynamics.png", dpi=140)
plt.show()
print("Saved → results/inv_grpo_training_dynamics.png")

---
## 7. Empirical Evaluation: M1 (Zero-Shot Baseline) vs. M6 (Inv-GRPO)

We evaluate the policy on the held-out test split of paired tasks ($L_0$ and $L_2$) under deterministic greedy decoding ($T=0.0$).

In [ ]:
evaluator = InvGRPOEvaluator(sandbox=reward_engine.sandbox)

# ── 1. Evaluate trained M6 (Inv-GRPO adapted model) FIRST ────────────────────
# trainer.model is already resident in GPU VRAM, so evaluate it first to avoid loading twice.
print("\n[M6 Eval] Evaluating Model M6 (Inv-GRPO trained) on 6 test pairs...")
m6_metrics = evaluator.evaluate_model(
    model=trainer.model,
    tokenizer=trainer.tokenizer,
    test_pairs=test_pairs,
    model_label="Model M6 (Inv-GRPO)",
)

# Free trainer model from VRAM before loading M1 baseline
del trainer.model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  M6 evaluation complete. VRAM freed.")

# ── 2. Evaluate M1 baseline (un-adapted base model) ──────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("\n[M1 Eval] Loading base model (no adapter) for baseline comparison...")
_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
_base_model_id = settings.models.student_model
_tok_m1 = AutoTokenizer.from_pretrained(_base_model_id, cache_dir=settings.storage.hf_cache_dir)
_tok_m1.pad_token = _tok_m1.eos_token
_mdl_m1 = AutoModelForCausalLM.from_pretrained(
    _base_model_id,
    quantization_config=_bnb_cfg,
    device_map="auto",
    cache_dir=settings.storage.hf_cache_dir,
)
print("  M1 Base model loaded. Running evaluation...")

m1_metrics = evaluator.evaluate_model(
    model=_mdl_m1,
    tokenizer=_tok_m1,
    test_pairs=test_pairs,
    model_label="Model M1 (Baseline)",
)

# Free M1 model
del _mdl_m1
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  M1 evaluation complete. VRAM freed.")

# ── 3. Plot publication-ready comparison ─────────────────────────────────────
evaluator.plot_comparison(
    m1_metrics=m1_metrics,
    m6_metrics=m6_metrics,
    save_path="results/inv_grpo_m1_vs_m6_comparison.png",
)
print("\n✅ Empirical Comparison Plot saved → results/inv_grpo_m1_vs_m6_comparison.png")


---
## 8. Summary of Findings & Thesis Defense Talking Points

### 🏆 Key Scientific Takeaways (for Dr. Ghada Soliman):
1. **Direct Mechanism for Invariance:** Standard RLVR evaluates prompts in isolation, rewarding accidental shortcut hits. Inv-GRPO explicitly ties credit assignment to cross-representation consistency.
2. **Suppression of Shortcut Weights:** The negative advantage assigned to Candidate 1 (Shortcut Mimic) directly pushes the policy gradients away from verbatim textbook memorization.
3. **Zero Inference Latency Overhead:** Unlike test-time majority voting or self-consistency loops (which multiply inference cost by $5\times$ or $10\times$), Inv-GRPO regularizes the weights at train-time, making Model M6 instantly deployable on Orange edge infrastructure.